# Challenge 11 Colab Ultra: stacking and blending

Esta notebook no entrena modelos base desde cero.

Su objetivo es combinar las salidas de las otras variantes avanzadas:

- `KNN cleaning`
- `SVM preprocessing`
- `signal features`

Para ello consume los artefactos:

- `oof_probabilities.csv`
- `test_probabilities.csv`

que quedan dentro del `output/` de cada notebook anterior.

## 0. Imports and global constants

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import gc
import itertools
import json
import platform
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.model_selection import ParameterSampler, StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression

try:
    import psutil
except ImportError:
    psutil = None

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["savefig.bbox"] = "tight"

RANDOM_STATE = 301655
VALID_SIZE = 0.20
NOTEBOOK_SLUG = "challenge_11_stacking_colab_ultra"

## 1. Create the Colab workspace

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files  # type: ignore
else:
    files = None

if IN_COLAB:
    WORKSPACE_ROOT = Path("/content/challenge_stacking_ultra_workspace")
else:
    cwd = Path.cwd().resolve()
    if (cwd / "challenge" / "data" / "training.csv").exists():
        WORKSPACE_ROOT = cwd / "challenge"
    elif (cwd / "data" / "training.csv").exists():
        WORKSPACE_ROOT = cwd
    else:
        WORKSPACE_ROOT = cwd / "challenge_stacking_ultra_workspace"

DATA_DIR = WORKSPACE_ROOT / "data"
TRAIN_PATH = DATA_DIR / "training.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_PATH = DATA_DIR / "sample.csv"

OUTPUT_ROOT = WORKSPACE_ROOT / "output"
PERSIST_ROOT = OUTPUT_ROOT / NOTEBOOK_SLUG
CHECKPOINT_DIR = PERSIST_ROOT / "checkpoints"
SUBMISSION_DIR = WORKSPACE_ROOT / "submissions"
EXPORT_DIR = WORKSPACE_ROOT / "exports"

for path in [WORKSPACE_ROOT, DATA_DIR, OUTPUT_ROOT, PERSIST_ROOT, CHECKPOINT_DIR, SUBMISSION_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("WORKSPACE_ROOT:", WORKSPACE_ROOT)
print("PERSIST_ROOT:", PERSIST_ROOT)

## 2. Inspect the workspace and expected files

In [ ]:
expected_files = {
    "training.csv": TRAIN_PATH,
    "test.csv": TEST_PATH,
    "sample.csv": SAMPLE_PATH,
}

print("Workspace directories:")
for path in [WORKSPACE_ROOT, DATA_DIR, OUTPUT_ROOT, PERSIST_ROOT, CHECKPOINT_DIR, SUBMISSION_DIR, EXPORT_DIR]:
    print("-", path)

print("\nData file status:")
for filename, path in expected_files.items():
    print(f"- {filename}: {'OK' if path.exists() else 'MISSING'} -> {path}")

## 3. Optional: upload the CSV files manually

In [ ]:
UPLOAD_DATA_FILES = False

if UPLOAD_DATA_FILES:
    if not IN_COLAB:
        raise RuntimeError("This upload helper is intended for Google Colab.")

    uploaded = files.upload()
    for original_name, file_bytes in uploaded.items():
        filename = Path(original_name).name
        target_path = DATA_DIR / filename
        target_path.write_bytes(file_bytes)
        print("Saved:", target_path)
else:
    print("Set UPLOAD_DATA_FILES = True if you want to upload training.csv, test.csv and sample.csv.")

## 4. Optional: restore resume bundles

In [ ]:
RESTORE_RESUME_BUNDLES = False

if RESTORE_RESUME_BUNDLES:
    if not IN_COLAB:
        raise RuntimeError("This restore helper is intended for Google Colab.")

    uploaded = files.upload()
    zip_names = [Path(name).name for name in uploaded if str(name).lower().endswith(".zip")]
    if not zip_names:
        raise ValueError("Upload at least one ZIP file when restoring artifacts.")

    for bundle_name in zip_names:
        bundle_path = EXPORT_DIR / bundle_name
        bundle_path.write_bytes(uploaded[bundle_name])
        with zipfile.ZipFile(bundle_path, "r") as zip_file:
            zip_file.extractall(WORKSPACE_ROOT)
        print("Restored:", bundle_path)
else:
    print("Set RESTORE_RESUME_BUNDLES = True if you want to restore one or more ZIP bundles.")

## 5. Checkpoint helpers and reusable utilities

In [ ]:
DEFAULT_BUNDLE_NAME = "challenge_11_stacking_colab_ultra_resume.zip"


def save_current_figure(filename: str) -> Path:
    path = PERSIST_ROOT / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    return path


def write_json_atomic(path: Path, payload: dict | list) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(payload, indent=2))
    tmp_path.replace(path)


def save_dataframe_atomic(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)


def read_dataframe(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def normalize_value(value):
    if isinstance(value, np.generic):
        return value.item()
    return value


def candidate_signature(params: dict) -> str:
    normalized = {key: normalize_value(value) for key, value in params.items()}
    return json.dumps(normalized, sort_keys=True)


def update_manifest(extra_payload: dict) -> dict:
    manifest_path = CHECKPOINT_DIR / "manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    manifest.update(extra_payload)
    write_json_atomic(manifest_path, manifest)
    return manifest


def create_resume_bundle(bundle_name: str = DEFAULT_BUNDLE_NAME, include_data: bool = True) -> Path:
    bundle_path = EXPORT_DIR / bundle_name
    if bundle_path.exists():
        bundle_path.unlink()

    paths_to_pack = []
    if include_data:
        paths_to_pack.append(DATA_DIR)
    paths_to_pack.extend([OUTPUT_ROOT, SUBMISSION_DIR])

    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
        for root_path in paths_to_pack:
            if not root_path.exists():
                continue
            for nested in root_path.rglob("*"):
                if nested.is_dir():
                    continue
                relative_path = nested.relative_to(WORKSPACE_ROOT)
                zip_file.write(nested, arcname=str(relative_path))

    return bundle_path


def checkpoint_housekeeping(stage_name: str, *, refresh_bundle: bool = False, include_data_in_bundle: bool = False) -> dict:
    payload = {"last_checkpoint_stage": stage_name}
    if refresh_bundle:
        bundle_path = create_resume_bundle(include_data=include_data_in_bundle)
        payload["resume_bundle_path"] = str(bundle_path)
        payload["resume_bundle_size_mb"] = round(bundle_path.stat().st_size / (1024 ** 2), 3)
    update_manifest(payload)
    return payload

## 6. Strategy-specific helpers

In [ ]:
def discover_probability_artifacts() -> tuple[dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:
    oof_paths = sorted(OUTPUT_ROOT.glob("*/oof_probabilities.csv"))
    test_paths = sorted(OUTPUT_ROOT.glob("*/test_probabilities.csv"))

    oof_map = {}
    test_map = {}
    for path in oof_paths:
        df = pd.read_csv(path).sort_values("id").reset_index(drop=True)
        source_name = str(df["source_model"].iloc[0]) if "source_model" in df.columns else path.parent.name
        oof_map[source_name] = df
    for path in test_paths:
        df = pd.read_csv(path).sort_values("id").reset_index(drop=True)
        source_name = str(df["source_model"].iloc[0]) if "source_model" in df.columns else path.parent.name
        test_map[source_name] = df
    return oof_map, test_map


def align_artifacts(oof_map: dict[str, pd.DataFrame], test_map: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, pd.DataFrame]:
    common_models = sorted(set(oof_map) & set(test_map))
    if len(common_models) < 2:
        raise RuntimeError("At least two base models with both OOF and test probabilities are required for stacking.")

    base_oof = oof_map[common_models[0]][["id", "y_true"]].copy()
    base_test = test_map[common_models[0]][["id"]].copy()
    for model_name in common_models:
        model_oof = oof_map[model_name][["id", "prob_1"]].rename(columns={"prob_1": model_name})
        model_test = test_map[model_name][["id", "prob_1"]].rename(columns={"prob_1": model_name})
        base_oof = base_oof.merge(model_oof, on="id", how="inner")
        base_test = base_test.merge(model_test, on="id", how="inner")
    return base_oof, base_test


def evaluate_average_subset(meta_df: pd.DataFrame, subset: list[str]) -> dict:
    probs = meta_df[subset].mean(axis=1).to_numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "meta_family": "average",
        "base_models": subset,
        "C": None,
        "cv_accuracy": float(accuracy_score(meta_df["y_true"], preds)),
    }


def evaluate_logreg_subset(meta_df: pd.DataFrame, subset: list[str], c_value: float) -> dict:
    X_meta = meta_df[subset].to_numpy(dtype=np.float32)
    y_meta = meta_df["y_true"].to_numpy(dtype=np.int8)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof_prob = np.zeros(len(meta_df), dtype=np.float32)
    for fit_idx, eval_idx in cv.split(X_meta, y_meta):
        model = LogisticRegression(C=c_value, max_iter=2000)
        model.fit(X_meta[fit_idx], y_meta[fit_idx])
        oof_prob[eval_idx] = model.predict_proba(X_meta[eval_idx])[:, 1].astype(np.float32)
    preds = (oof_prob >= 0.5).astype(int)
    return {
        "meta_family": "logreg",
        "base_models": subset,
        "C": c_value,
        "cv_accuracy": float(accuracy_score(y_meta, preds)),
    }

## 7. Validate that the required CSV files are present

In [ ]:
missing_files = [str(path) for path in [TRAIN_PATH, TEST_PATH, SAMPLE_PATH] if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required data files. Upload training.csv, test.csv and sample.csv first or restore a resume ZIP. "
        f"Missing: {missing_files}"
    )

print("All required CSV files are present.")

## 8. Runtime inspection and search budget

In [ ]:
SEARCH_PROFILE = "aggressive"
RUN_DISCOVERY = True
RUN_SEARCH = True
TRAIN_FINAL_MODEL = True

META_C_VALUES = [0.25, 1.0, 4.0, 16.0]
print({"python": platform.python_version(), "sklearn": sklearn.__version__})

## 9. Load the challenge data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_df = pd.read_csv(SAMPLE_PATH)

feature_names = [column for column in train_df.columns if column not in {"id", "class"}]
train_ids = train_df["id"].to_numpy()
test_ids = test_df["id"].to_numpy()

X_full = train_df[feature_names].astype(np.float32).to_numpy()
y_full = train_df["class"].astype(np.int8).to_numpy()
X_test_full = test_df[feature_names].astype(np.float32).to_numpy()

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Class balance:", train_df["class"].value_counts().sort_index().to_dict())
print("Missing values in training:", int(train_df.isna().sum().sum()))
print("Duplicated rows in training:", int(train_df.duplicated().sum()))

X_train, X_valid, y_train, y_valid = train_test_split(
    X_full,
    y_full,
    test_size=VALID_SIZE,
    stratify=y_full,
    random_state=RANDOM_STATE,
)

print("Train split:", X_train.shape, y_train.shape)
print("Validation split:", X_valid.shape, y_valid.shape)

## 10. Stage 1

In [ ]:
if RUN_DISCOVERY:
    oof_map, test_map = discover_probability_artifacts()
    print("Discovered OOF artifacts:", sorted(oof_map))
    print("Discovered test artifacts:", sorted(test_map))
else:
    oof_map, test_map = {}, {}

meta_oof_df, meta_test_df = align_artifacts(oof_map, test_map)
print("Meta OOF shape:", meta_oof_df.shape)
print("Meta test shape:", meta_test_df.shape)
display(meta_oof_df.head())

corr_df = meta_oof_df.drop(columns=["id", "y_true"]).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_df, annot=True, fmt=".3f", cmap="viridis")
plt.title("Correlation of base-model OOF probabilities")
save_current_figure("oof_probability_correlation.png")
checkpoint_housekeeping("stage1_artifact_discovery_complete", refresh_bundle=True, include_data_in_bundle=False)

## 11. Stage 2

In [ ]:
search_results_path = CHECKPOINT_DIR / "stacking_search_results.csv"
search_df = read_dataframe(search_results_path)
completed_signatures = set(search_df["signature"]) if not search_df.empty else set()

base_model_names = [column for column in meta_oof_df.columns if column not in {"id", "y_true"}]
search_candidates = []
next_index = 0
for subset_size in range(2, len(base_model_names) + 1):
    for subset in itertools.combinations(base_model_names, subset_size):
        subset_list = list(subset)
        avg_signature = json.dumps({"meta_family": "average", "base_models": subset_list}, sort_keys=True)
        search_candidates.append({"candidate_id": f"blend_{next_index:03d}", "signature": avg_signature, "meta_family": "average", "base_models": subset_list, "C": None})
        next_index += 1
        for c_value in META_C_VALUES:
            signature = json.dumps({"meta_family": "logreg", "base_models": subset_list, "C": c_value}, sort_keys=True)
            search_candidates.append({"candidate_id": f"blend_{next_index:03d}", "signature": signature, "meta_family": "logreg", "base_models": subset_list, "C": c_value})
            next_index += 1

print("Stacking candidates:", len(search_candidates))
print("Already completed:", len(completed_signatures))

if RUN_SEARCH:
    pending = [candidate for candidate in search_candidates if candidate["signature"] not in completed_signatures]
    for candidate in tqdm(pending, desc="Stacking candidates"):
        if candidate["meta_family"] == "average":
            result = evaluate_average_subset(meta_oof_df, candidate["base_models"])
        else:
            result = evaluate_logreg_subset(meta_oof_df, candidate["base_models"], float(candidate["C"]))
        row = {
            "candidate_id": candidate["candidate_id"],
            "signature": candidate["signature"],
            "meta_family": result["meta_family"],
            "base_models_json": json.dumps(result["base_models"]),
            "C": result["C"],
            "cv_accuracy": result["cv_accuracy"],
        }
        search_df = pd.concat([search_df, pd.DataFrame([row])], ignore_index=True) if not search_df.empty else pd.DataFrame([row])
        search_df = search_df.sort_values(["cv_accuracy"], ascending=[False]).reset_index(drop=True)
        save_dataframe_atomic(search_df, search_results_path)

search_df = read_dataframe(search_results_path)
display(search_df.head(20))
checkpoint_housekeeping("stage2_stacking_search_complete", refresh_bundle=True, include_data_in_bundle=False)

## 12. Stage 3

In [ ]:
stacking_summary = search_df.copy()

## 13. Diagnostic plots

In [ ]:
if not search_df.empty:
    plt.figure(figsize=(12, 6))
    plot_df = search_df.head(20).copy()
    sns.barplot(data=plot_df, x="candidate_id", y="cv_accuracy", hue="meta_family")
    plt.xticks(rotation=75, ha="right")
    plt.title("Top stacking candidates")
    save_current_figure("top_stacking_candidates.png")

## 14. Final model, OOF artifacts and submission

In [ ]:
if search_df.empty:
    raise RuntimeError("No stacking search results are available.")

best_row = search_df.iloc[0].to_dict()
base_models = json.loads(best_row["base_models_json"])
meta_family = best_row["meta_family"]

X_meta_train = meta_oof_df[base_models].to_numpy(dtype=np.float32)
y_meta = meta_oof_df["y_true"].to_numpy(dtype=np.int8)
X_meta_test = meta_test_df[base_models].to_numpy(dtype=np.float32)

if meta_family == "average":
    train_prob = X_meta_train.mean(axis=1)
    test_prob = X_meta_test.mean(axis=1)
else:
    meta_model = LogisticRegression(C=float(best_row["C"]), max_iter=2000)
    meta_model.fit(X_meta_train, y_meta)
    train_prob = meta_model.predict_proba(X_meta_train)[:, 1]
    test_prob = meta_model.predict_proba(X_meta_test)[:, 1]

train_pred = (train_prob >= 0.5).astype(int)
test_pred = (test_prob >= 0.5).astype(int)

cm = confusion_matrix(y_meta, train_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", colorbar=False)
plt.title(f"Meta-model training confusion matrix - accuracy={accuracy_score(y_meta, train_pred):.4f}")
save_current_figure("meta_training_confusion_matrix.png")

submission_df = sample_df.copy()
submission_df["class"] = test_pred.astype(int)
submission_path = SUBMISSION_DIR / "challenge_11_stacking_colab_ultra_submission.csv"
submission_df.to_csv(submission_path, index=False)

summary_payload = {
    "model_name": "Stacking / blending meta-model",
    "model_key": "stacking",
    "notebook_slug": NOTEBOOK_SLUG,
    "strategy": "colab_ultra_stacking",
    "best_meta_family": meta_family,
    "best_base_models": base_models,
    "best_C": None if pd.isna(best_row["C"]) else float(best_row["C"]),
    "meta_training_accuracy": float(accuracy_score(y_meta, train_pred)),
    "submission_path": str(submission_path),
    "workspace_root": str(WORKSPACE_ROOT),
    "persist_root": str(PERSIST_ROOT),
}
write_json_atomic(PERSIST_ROOT / "summary.json", summary_payload)
checkpoint_housekeeping("final_model_complete", refresh_bundle=True, include_data_in_bundle=False)

print("Best stacking candidate:", json.dumps(summary_payload, indent=2))
print("Submission path:", submission_path)

## 15. Optional: export and download a manual resume bundle

In [ ]:
INCLUDE_DATA_IN_MANUAL_BUNDLE = True
DOWNLOAD_BUNDLE_NOW = False

bundle_path = create_resume_bundle(include_data=INCLUDE_DATA_IN_MANUAL_BUNDLE)
print("Resume bundle saved to:", bundle_path)

if DOWNLOAD_BUNDLE_NOW and IN_COLAB:
    files.download(str(bundle_path))

## Notes

Esta notebook debe ejecutarse despues de que al menos dos notebooks base hayan generado:

- `oof_probabilities.csv`
- `test_probabilities.csv`

Lo normal es restaurar los ZIP de esas corridas dentro del mismo workspace antes de correr el stacking.